# Disruption demo: Reconstruct figure 1 of Montes, et al, 2019

## LLM Model

First pick the LLM model.  The following command assumes the API key is in your `.env` file in your persistent directory under the variable `AMSC_LLM_KEY`

In [2]:
%model --url https://api.i2-core.american-science-cloud.org --api-key-env AMSC_LLM_KEY --name claude-sonnet-4-6

model,claude:claude-sonnet-4-6
provider,claude
endpoint,https://api.i2-core.american-science-cloud.org
api key,AMSC_LLM_KEY is set
configured,"claude, nrp"
config file,/home/jovyan/.deepagents/config.toml


## FDP MCP server

Now set up the FDP MCP server.

In [3]:
%%mcp
{
  "mcpServers": {
    "toksearch-skills": {
      "command": "python",
      "args": ["-m", "fdp", "-D", "d3d", "run", "python", "-m", "toksearch.llm.mcp"]
    }
  }
}

## Ask to recreate figure 1 from Montes et al, 2019

The prompt will be rather long and split into parts, to get a good level of detail.

In [ ]:
%%ask

# The ask:
Rebuild the Alcator C-Mod and DIII-D loop-voltage panels from figure 1 of Montes et al,
Nuclear Fusion 59 096015 (2019).

Take the shots that disrupted during flat-top -- meaning every shot whose
last flat-top sample falls inside the 250 ms window before the quench, not
only those still classified as flat-top at the instant of the disruption --
and slice the 250 ms immediately before the current quench. Plot loop
voltage, signed by the direction of the plasma current, against time until
disruption: a two-dimensional density over all samples, with the median and
the 10th and 90th percentiles drawn over it.

Beside that, compare the distribution of the same signed loop voltage between
two populations: the disruptive shots during their final 250 ms, and the
non-disruptive shots during their flat-top phase only.  Plot it as a horizontal 
histogram so the y axes line up with the vloop vs time before disruption plot.

# The setup
For DIII-D, ignore the disruption data in the lakehouse.  Reprocess the necessary data with disruption-py.

Keep track of how much time it takes to fetch data versus doing any processing versus querying the LLM.  

Try to perform any fetches in parallel.

# The plotting
State how many shots and how many samples stand behind each panel.

For the density panel, so that it can be set beside the published figure:
- normalise within each time column, so the color is the share of samples at
  that time rather than a raw count, and say so in the caption
- use a white-to-red sequential colormap on a linear scale, and keep the
  colorbar and the legends clear of the panel titles and the data
- set the color scale to the range the shares actually span, not 0 to 1, so the
  density is visible
- match the published figure's voltage axis, 0 to 3.5 V, and state how many
  samples fall outside it and how far
- draw the non-disruptive median as a single horizontal reference line across
  the panel

Check the sampling cadence before plotting rather than assuming it is uniform,
and say what you find.

Reading the cmod-disruption and d3d-disruption-compute skill files, plus their Learnings.md, in parallel.

The read_skill tool uses a different index. Let me read the SKILL.md files directly from the filesystem.

Now reading the Learnings.md files for cmod-disruption and d3d-disruption-compute and d3d-shot-fetcher (which is listed as having Learnings.md).

Excellent — learnings applied. Now let me check what parameters are available in the cmod index, especially for loop voltage and discharge phase, then set up a plan for both devices.

Writing an exploration script to probe the C-Mod index and also check what shots are available in the D3D index.

...

Reading the rest of d3d_disruption_compute.py to understand timebase_covering, add_time_until_disrupt, and working_dir.

Now I have a complete picture of both APIs. Let me now write the full data-fetching script. The strategy:
1. C-Mod: Use `window_before_disruption` for disruptive shots + fetch flat-top samples from non-disruptive shots
2. D3D: Use `disruption_times()` to identify disrupted shots, then `compute()` with loop voltage + ip + phase info

Let me first explore what D3D shots are available and what parameters I need from D3D.

The D3D stored index has only 20 shots (10 disrupted, 10 non-disrupted). The instructions say to ignore the disruption data in the lakehouse and reprocess with disruption-py. I need a larger shot list. Let me check what shots are available in d3drdb (disruptions table) which `disruption_times()` pulls from.

There are 24,219 disrupted shots. The task says to reprocess D3D with disruption-py. That's too many to run all at once. The paper (Montes et al 2019) used specific shots. Let me look at the stored D3D index shots and check what non-disruptive shots I can find. I'll use a practical approach: run disruption-py on the shots from the stored D3D index (10 disrupted + 10 non-disrupted = 20 shots), since the instructions say to reprocess rather than read from the lakehouse. Let me also check if there are more shots available from the stored index and see if I can query a specific era of shots.

Good — disruption-py is available. The stored D3D index has 10 disrupted + 10 non-disrupted shots. Per the instructions, I need to reprocess these with disruption-py. The shot range is 194000-199687. 

The task says to get shots that "disrupted during flat-top" — the 250ms criterion. I'll run disruption-py on all 20 stored shots plus query d3drdb for more non-disruptive shots and a larger sample of disruptive shots in a similar range, to get a more meaningful dataset.

Let me check what shots are available from d3drdb.disruptions in a recent era and find non-disruptive shots too.